# The below script processes tables with multirow cells

In [ ]:
import os
import xml.etree.ElementTree as ET

def parse_tei_to_markdown_with_metadata(xml_file_path, output_md_file):
    tree = ET.parse(xml_file_path)
    root = tree.getroot()

    footnotes = []  # Store footnotes to write at the bottom
    footnote_counter = 1

    def process_text_elements(parent_elem, include_notes=True):
        """Extracts and formats text with child elements like emph, q, geogName, name (ship names), and notes."""
        text_parts = []
        if parent_elem.text:
            text_parts.append(parent_elem.text.strip())

        for child in parent_elem:
            # Handle specific element formatting
            if child.tag == "emph":
                text_parts.append(f"*{child.text.strip()}*" if child.text else "")
            elif child.tag == "q":
                text_parts.append(f"'{child.text.strip()}'" if child.text else "")
            elif child.tag == "date" and "value" in child.attrib:
                date_text = child.text.strip() if child.text else ""
                text_parts.append(f" {{{child.attrib['value']}}} {date_text}")
            elif child.tag == "geogName":
                text_parts.append(f' <span style="border-bottom: 2px dotted #FF0000;">{child.text.strip()}</span> ' if child.text else "")
            elif child.tag == "name" and child.attrib.get("type") == "geographical":
                text_parts.append(f' <span style="border-bottom: 2px dotted #FF0000;">{child.text.strip()}</span> ' if child.text else "")
            elif child.tag == "name" and child.attrib.get("type") == "scheepsnaam":
                text_parts.append(f' <span style="border-bottom: 2px dotted #0000FF;">{child.text.strip()}</span> ' if child.text else "")
            elif child.tag == "name" and child.attrib.get("type") == "person":
                text_parts.append(f' <span style="border-bottom: 2px dotted #00FF00;">{child.text.strip()}</span> ' if child.text else "")
            elif child.tag == "signed":
                signed_text = process_text_elements(child)
                text_parts.append(f"[Signed:] {signed_text}")
            elif child.tag == "table":
                table_md = process_table_element(child)
                text_parts.append(table_md)
            
            # Handle footnotes consistently across all elements
            if child.tag == "note" and include_notes:
                nonlocal footnote_counter
                note_content = process_text_elements(child, include_notes=False)
                text_parts.append(f"[^{footnote_counter}] ")  # Insert footnote reference directly
                footnotes.append(f"[^{footnote_counter}]: {note_content}")
                footnote_counter += 1

            # Handle text following child elements (tail text)
            if child.tail:
                text_parts.append(child.tail.strip())

        return "".join(text_parts)  # Avoid leading/trailing spaces or line breaks

    def process_table_element(table_elem):
        """Processes tables and returns an HTML representation, handling <head> correctly and supporting row-spanning cells."""
        table_md = ["<table>"]

        # Handle <head> as a title row if it exists
        header_tag = table_elem.find("head")
        head_content = ""

        if header_tag is not None:
            head_content = ''.join(header_tag.itertext()).strip()
            # Remove the head tag to prevent duplication
            table_elem.remove(header_tag)

        # Add <thead> if head_content is present
        if head_content:
            table_md.append(f"  <thead><tr><td colspan='100%'>{head_content}</td></tr></thead>")

        table_md.append("  <tbody>")

        rows = table_elem.findall("row")
        row_span_map = {}  # Track spanning cells

        # Process table rows
        for row_index, row in enumerate(rows):
            table_md.append("    <tr>")
            cells = row.findall("cell")
            col_index = 0

            for cell in cells:
                # Skip columns already covered by row-spanning cells
                while row_span_map.get((row_index, col_index), 0) > 0:
                    row_span_map[(row_index, col_index)] -= 1
                    col_index += 1

                # Process the current cell
                cell_content = process_text_elements(cell).strip() or '&nbsp;'
                rowspan = int(cell.attrib.get("rows", "1"))

                # Add rowspan attribute and center text vertically if necessary
                span_attr = f" rowspan='{rowspan}'" if rowspan > 1 else ""
                style_attr = " style='vertical-align: middle;'" if rowspan > 1 else ""

                table_md.append(f"      <td{span_attr}{style_attr}>{cell_content}</td>")

                # Mark row-spanning cells in the map
                for i in range(1, rowspan):
                    row_span_map[(row_index + i, col_index)] = rowspan - i

                col_index += 1

            table_md.append("    </tr>")

        table_md.append("  </tbody>")
        table_md.append("</table>")

        return "\n".join(table_md) + "\n"

    with open(output_md_file, "w", encoding="utf-8") as md_file:
        for elem in root.iter():
            if elem.tag == "div" and elem.attrib.get("n"):
                md_file.write(f"## {elem.attrib['n']}\n\n")
            if elem.tag == "head":
                head_text = process_text_elements(elem)
                md_file.write(f"{head_text}\n\n")
            if elem.tag == "present":
                present_text = process_text_elements(elem)
                md_file.write(f"{present_text}\n\n")
            if elem.tag == "p":
                paragraph_text = process_text_elements(elem)
                if paragraph_text:
                    md_file.write(paragraph_text + "\n\n")
            if elem.tag == "q":
                quote_text = process_text_elements(elem)
                if quote_text:
                    md_file.write(f"'{quote_text}'\n\n")
            if elem.tag == "signed":
                signed_text = process_text_elements(elem)
                if signed_text:
                    md_file.write(f"[Signed:] {signed_text}\n\n")
            if elem.tag == "table":  # Handle tables *directly*
                table_md = process_table_element(elem)
                md_file.write(table_md + "\n")

        if footnotes:
            md_file.write("\n---\n\n ## Footnotes \n")
            for footnote in footnotes:
                md_file.write(footnote + "\n\n")


def process_all_xml_files(input_dir, output_dir):
    """Processes all XML files in a directory and its subdirectories, writing output as Markdown."""
    for root_dir, _, files in os.walk(input_dir):
        for file_name in files:
            if file_name.endswith(".xml"):
                xml_file_path = os.path.join(root_dir, file_name)

                # Create corresponding output path
                relative_path = os.path.relpath(xml_file_path, input_dir)
                md_file_path = os.path.join(output_dir, os.path.splitext(relative_path)[0] + ".md")

                os.makedirs(os.path.dirname(md_file_path), exist_ok=True)

                # Parse and write markdown
                parse_tei_to_markdown_with_metadata(xml_file_path, md_file_path)
                print(f"Processed: {xml_file_path} -> {md_file_path}")

# Example usage
input_dir = "/path/to/TANAP_transcriptions"
output_dir = "/path/to/TANAP_transcriptions_md_new"
process_all_xml_files(input_dir, output_dir)


## The next cells processes the glossary

In [ ]:
import xml.etree.ElementTree as ET
import re

def extract_glossary_entries(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    glossary_entries = []
    current_letter = None

    # Helper function to extract text with emphasis handling
    def extract_text_with_emph(parent):
        text_parts = []
        
        if parent.text:
            text_parts.append(parent.text.strip())
        
        for child in parent:
            if child.tag == "emph":
                emph_text = f"*{child.text.strip()}*" if child.text else ""
                tail_text = child.tail.strip() if child.tail else ""
                
                # Avoid adding spaces before punctuation
                if re.match(r"^[.,;:!?]", tail_text):
                    text_parts.append(f"{emph_text}{tail_text}")
                else:
                    text_parts.append(f"{emph_text} {tail_text}".strip())
        
        return " ".join(text_parts).strip()

    for div in root.findall(".//div"):
        head = div.find(".//head")
        if head is not None and head.text:
            letter = head.text.strip()
            if letter != current_letter:
                glossary_entries.append(f"## {letter}")
                current_letter = letter

            for entry in div.findall(".//entry"):
                forms_and_senses = []

                form = entry.find(".//form")
                sense = entry.find(".//sense")

                if form is not None:
                    form_text = extract_text_with_emph(form)
                    if sense is not None:
                        sense_text = extract_text_with_emph(sense)
                        forms_and_senses.append(f"**{form_text}**: {sense_text}")
                    else:
                        forms_and_senses.append(f"**{form_text}**")

                if forms_and_senses:
                    glossary_entries.append("\n".join(forms_and_senses))

    return "\n\n".join(glossary_entries)

# Process the glossary file
glossary_output = extract_glossary_entries('/path/to/TANAP_transcriptions/Council of Policy/Biblio+glos/Glos_xml_v1.1.xml')

# Check the output before writing
print(f"Generated output (first 500 characters): {glossary_output[:500]}...")  # Display the first 500 chars of the output for debugging

# Write the result to a Markdown file
if glossary_output:
    with open('/path/to/TANAP_transcriptions_md_new/glossary_output.md', "w") as md_file:
        md_file.write(glossary_output)
    print("Markdown file 'glossary_output.md' has been created.")
else:
    print("No glossary entries were extracted. Check the XML structure.")


## The cell below transforms the Bibliografie_xml_v1.0.xml file

In [ ]:
from lxml import etree

def process_bibliografie_xml(file_path):
    tree = etree.parse(file_path)
    root = tree.getroot()

    md_output = ["## Bibliography\n"]
    
    # Find all <item> entries
    items = root.xpath('//item')
    for item in items:
        author = item.xpath('.//author/text()')
        title = item.xpath('.//title/text()')
        pub_place = item.xpath('.//pubPlace/text()')
        date = item.xpath('.//date/text()')
        publisher = item.xpath('.//publisher/text()')

        # Format each bibliographic entry
        author_text = author[0] if author else ""
        title_text = title[0] if title else "Untitled"
        pub_place_text = pub_place[0] if pub_place else "Unknown Place"
        date_text = date[0] if date else "Unknown Date"
        publisher_text = publisher[0] if publisher else ""

        # Create the formatted bibliography entry
        md_output.append(f"- {author_text} *{title_text}* {pub_place_text} {publisher_text} {date_text}\n")
    
    return ''.join(md_output)

# Example usage
bib_md = process_bibliografie_xml('/path/to/TANAP_transcriptions/Council of Policy/Biblio+glos/Bibliografie_xml_v1.0.xml')
with open('/path/to/TANAP_transcriptions_md_new/bibliography_output.md', 'w') as f:
    f.write(bib_md)
